### 2.1 理论计算题

**已知数据：**
*   字符序列为 `"ababc"`
*   词汇表 $V = \{\text{'a'}, \text{'b'}, \text{'c'}\}$，即 $|V| = 3$。

**一阶转移统计：**
在序列 `"ababc"` 中，所有相邻字符组成的转移对为：`('a', 'b')`, `('b', 'a')`, `('a', 'b')`, `('b', 'c')`。
由此统计出从状态 `'b'` 出发的转移次数：
*   $C(\text{'b'}, \text{'a'}) = 1$ （出现一次 `'ba'`）
*   $C(\text{'b'}, \text{'b'}) = 0$ （未出现 `'bb'`）
*   $C(\text{'b'}, \text{'c'}) = 1$ （出现一次 `'bc'`）
*   从状态 `'b'` 出发的总转移次数 $C(\text{'b'}) = 1 + 0 + 1 = 2$。

使用拉普拉斯平滑（加 1 平滑）估计条件概率公式：
$$ P(x_t = j \mid x_{t-1} = i) = \frac{C(i, j) + 1}{C(i) + |V|} $$

#### 1. 计算 $P(\text{'a'} \mid \text{'b'})$
$$ P(\text{'a'} \mid \text{'b'}) = \frac{C(\text{'b'}, \text{'a'}) + 1}{C(\text{'b'}) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4 $$

#### 2. 计算 $P(\text{'c'} \mid \text{'b'})$
$$ P(\text{'c'} \mid \text{'b'}) = \frac{C(\text{'b'}, \text{'c'}) + 1}{C(\text{'b'}) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4 $$

**答：** 估计后的条件概率分别为 $P(\text{'a'} \mid \text{'b'}) = 0.4$，$P(\text{'c'} \mid \text{'b'}) = 0.4$。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理及滑动窗口生成
    """
    # 1. 将文本转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按频数降序，频数相同按字母排序，分配 ID 自 0 开始）
    word_counts = Counter(words)
    sorted_words = [word for word, _ in sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))]
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    L = len(words)
    for i in range(L - n + 1):
        features.append(words[i:i+n])
        if i + n < L:
            labels.append(words[i+n])
        else:
            labels.append(None)  # 若无后续词则为 None
            
    return vocab, (features, labels)

# 验证测试
text = "The time machine"
vocab, (features, labels) = preprocess_text(text, n=2)
print("词汇表字典:", vocab)
print("特征列表:", features)
print("标签列表:", labels)

词汇表字典: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time'], ['time', 'machine']]
标签列表: ['machine', None]


### 3.1 理论计算题

**已知线性 RNN 模型：**
$$ h_t = W_{hh} h_{t-1} + W_{hx} x_t $$
$$ o_t = W_{oh} h_t $$
损失函数采用平方损失：$L = \frac{1}{2} \sum_{t=1}^T (o_t - y_t)^2$。

#### 1. 梯度表达式 $\frac{\partial L}{\partial W_{hh}}$ 推导：
根据时间反向传播（BPTT），总损失对 $W_{hh}$ 的偏导数是各个时间步损失偏导数的累加：
$$ \frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L_t}{\partial W_{hh}} $$
对于特定的时间步 $t$，由于 $h_t$ 依赖于过去的隐藏状态，利用链式法则展开：
$$ \frac{\partial L_t}{\partial W_{hh}} = \sum_{k=1}^t \frac{\partial L_t}{\partial o_t} \frac{\partial o_t}{\partial h_t} \frac{\partial h_t}{\partial h_k} \frac{\partial h_k}{\partial W_{hh}} $$

其中各项偏导分别为：
*   $\frac{\partial L_t}{\partial o_t} = (o_t - y_t)$
*   $\frac{\partial o_t}{\partial h_t} = W_{oh}$
*   对于线性网络，有 $\frac{\partial h_j}{\partial h_{j-1}} = W_{hh}$，因此：
    $$ \frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^t \frac{\partial h_j}{\partial h_{j-1}} = W_{hh}^{t-k} $$
*   直接偏导项 $\frac{\partial h_k}{\partial W_{hh}} = h_{k-1}$

结合上述，写成矩阵形式的梯度表达式为：
$$ \frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^t (W_{hh}^{t-k})^T W_{oh}^T (o_t - y_t) h_{k-1}^T $$

---

#### 2. 梯度消失或爆炸的条件：
反向传播的关键项在于矩阵的幂次项：$(W_{hh}^{t-k})^T$。
设 $\rho(W_{hh})$ 为矩阵 $W_{hh}$ 的谱半径（最大特征值绝对值）：
*   **梯度消失条件：** 当 $\rho(W_{hh}) < 1$ 且时间间隔 $t-k$ 很大时，$W_{hh}^{t-k} \to 0$ 呈指数级衰减。这导致久远历史步（较小 $k$ 处）传回来的梯度几乎为 $0$，模型失去长周期记忆能力。
*   **梯度爆炸条件：** 当 $\rho(W_{hh}) > 1$ 且 $t-k$ 较大时，$W_{hh}^{t-k} \to \infty$ 呈指数级膨胀，造成梯度极其庞大，进而引发数值溢出。

In [3]:
import torch

def rnn_step_forward_backward(x_t, h_prev, W_hx, W_hh, b_h, dh_next):
    """
    单步 RNN 单元前向与反向传播 (采用右乘习惯: h_t = tanh(x_t * W_hx + h_prev * W_hh + b_h))
    """
    # 1. 前向传播
    z_t = torch.matmul(x_t, W_hx) + torch.matmul(h_prev, W_hh) + b_h
    h_t = torch.tanh(z_t)
    
    # 2. 反向传播
    # tanh 的导数为 1 - tanh^2(z)
    dz_t = dh_next * (1.0 - h_t ** 2)
    
    dx_t = torch.matmul(dz_t, W_hx.T)
    dh_prev = torch.matmul(dz_t, W_hh.T)
    
    dW_hx = torch.matmul(x_t.T, dz_t)
    dW_hh = torch.matmul(h_prev.T, dz_t)
    db_h = dz_t.sum(dim=0)
    
    return h_t, (dx_t, dh_prev, dW_hx, dW_hh, db_h)

# 验证测试
batch, in_sz, hid_sz = 3, 4, 5
x_t = torch.randn(batch, in_sz)
h_prev = torch.randn(batch, hid_sz)
W_hx = torch.randn(in_sz, hid_sz)
W_hh = torch.randn(hid_sz, hid_sz)
b_h = torch.randn(hid_sz)
dh_next = torch.randn(batch, hid_sz)

h_t, grads = rnn_step_forward_backward(x_t, h_prev, W_hx, W_hh, b_h, dh_next)
print("前向隐藏状态 h_t 形状:", h_t.shape)
print("反向 dx_t 形状:", grads[0].shape)
print("反向 dW_hh 形状:", grads[3].shape)

前向隐藏状态 h_t 形状: torch.Size([3, 5])
反向 dx_t 形状: torch.Size([3, 4])
反向 dW_hh 形状: torch.Size([5, 5])


### 4.1 理论计算题

**条件：**
*   深度双向 RNN，共 $L$ 层。
*   每层的前向和反向隐藏单元数均为 $H$。
*   第一层输入维度为 $D$。
*   最后输出层输出维度为 $O$。

**参数量逐层拆解计算（含权重与偏置）：**

1.  **第一层（Layer 1）：**
    *   前向 RNN：输入维度 $D$，隐藏维度 $H$。参数量：$W_{hx} (H \times D)$ + $W_{hh} (H \times H)$ + $b_h (H) = HD + H^2 + H$。
    *   反向 RNN：输入维度 $D$，隐藏维度 $H$。参数量：同样为 $HD + H^2 + H$。
    *   第一层总参数量：$2(HD + H^2 + H) = 2HD + 2H^2 + 2H$。
    *   *注：第一层的双向输出合并后，作为第二层输入的特征维度为 $2H$。*

2.  **后续层（Layer $l$, 其中 $2 \le l \le L$）：**
    *   前向 RNN：输入维度为前一层的拼接输出 $2H$，隐藏维度 $H$。参数量：$W_{hx} (H \times 2H)$ + $W_{hh} (H \times H)$ + $b_h (H) = 2H^2 + H^2 + H = 3H^2 + H$。
    *   反向 RNN：参数量相同，为 $3H^2 + H$。
    *   每层双向总参数量：$2(3H^2 + H) = 6H^2 + 2H$。
    *   后续 $L-1$ 层累计参数量：$(L-1)(6H^2 + 2H)$。

3.  **最终输出层：**
    *   将最后一层双向输出拼接（维度 $2H$）映射到输出空间 $O$。
    *   参数量：$W_{ho} (O \times 2H)$ + $b_o (O) = 2HO + O$。

**总参数量公式表达式：**
$$ \text{Total Params} = (2HD + 2H^2 + 2H) + (L-1)(6H^2 + 2H) + (2HO + O) $$
化简整理后得到：
$$ \text{Total Params} = 2H(D + O) + (6L - 4)H^2 + 2LH + O $$

In [4]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    """
    双向 RNN 编码器
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 使用 PyTorch 官方单层双向 RNN
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers=1, bidirectional=True)
        
    def forward(self, X):
        # X 形状: (seq_len, batch, input_dim)
        outputs, h_n = self.rnn(X)
        # outputs 形状: (seq_len, batch, 2 * hidden_dim)
        
        # h_n 形状: (2, batch, hidden_dim)
        # h_n[0] 为前向最终步状态，h_n[1] 为后向最终步状态，拼接它们
        final_hidden = torch.cat((h_n[0], h_n[1]), dim=-1) # 形状: (batch, 2 * hidden_dim)
        
        return outputs, final_hidden

# 验证测试
seq_len, batch, in_dim, hid_dim = 7, 2, 8, 16
encoder = BiRNNEncoder(in_dim, hid_dim)
X = torch.randn(seq_len, batch, in_dim)
outputs, final_hidden = encoder(X)

print("每个时间步拼接隐藏状态输出形状:", outputs.shape)
print("最终步拼接隐藏状态输出形状:", final_hidden.shape)

每个时间步拼接隐藏状态输出形状: torch.Size([7, 2, 32])
最终步拼接隐藏状态输出形状: torch.Size([2, 32])


### 5.1 理论计算题

**Skip-gram 负采样损失函数推导：**

在 Skip-gram 负采样中，对于给定的中心词 $w_c$ 和真实上下文词 $w_o$，我们将问题转换为二分类：最大化真实对 $(w_c, w_o)$ 的共现概率，同时最小化 $K$ 个来自噪声分布 $P_n(w)$ 的噪声词 $w_{n_k}$ 与中心词的共现概率。

设中心词向量为 $\mathbf{v}_c$，真实上下文词向量为 $\mathbf{u}_o$，负样本词向量为 $\mathbf{u}_{n_k}$。

1.  **噪声分布采样方法：**
    在 Word2vec 中，负样本采用修改后的一元词频（Unigram）分布进行采样。为了提升低频词被采样的概率，通常对其乘以乘方 $3/4$：
    $$ P_n(w) = \frac{U(w)^{3/4}}{\sum_{w'} U(w')^{3/4}} $$
    其中 $U(w)$ 为词 $w$ 在语料中的真实出现频率。

2.  **损失函数目标函数：**
    根据最大对数似然原则，极大化真实样本概率并极小化负样本概率，定义**单个样本对的损失函数**（即最小化目标负对数似然）为：
    $$ L = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^K \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) $$
    其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 是 Sigmoid 函数。

In [5]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, targets, W, W_out):
    """
    CBOW 模型的前向传播与完整 Softmax 交叉熵损失计算
    """
    # 1. 提取上下文词的嵌入向量并计算均值作为隐藏层
    # context_indices 形状: (batch_size, context_size)
    # W 形状: (V, d)
    embeds = W[context_indices]       # 形状: (batch_size, context_size, d)
    h = embeds.mean(dim=1)            # 形状: (batch_size, d)
    
    # 2. 投影到输出权重空间
    # W_out 形状: (d, V)
    logits = torch.matmul(h, W_out)   # 形状: (batch_size, V)
    
    # 3. 计算多分类交叉熵损失
    loss = F.cross_entropy(logits, targets)
    return loss

# 验证测试
batch, V, d, context_size = 3, 100, 32, 4
W = torch.randn(V, d)
W_out = torch.randn(d, V)
context_indices = torch.randint(0, V, (batch, context_size))
targets = torch.randint(0, V, (batch,))

loss = cbow_forward_loss(context_indices, targets, W, W_out)
print("CBOW 交叉熵 Loss 值为:", loss.item())

CBOW 交叉熵 Loss 值为: 8.410523414611816


### 6.1 理论计算题

**已知数据：**
查询矩阵 $Q \in \mathbb{R}^{2\times4}$，键矩阵 $K \in \mathbb{R}^{3\times4}$，值矩阵 $V \in \mathbb{R}^{3\times5}$。
缩放因子 $\sqrt{d_k} = \sqrt{4} = 2$。

**缩放点积注意力计算三步走过程：**

1.  **第一步：计算得分矩阵（Score Matrix） $A$**
    $$ A = \frac{Q K^T}{\sqrt{d_k}} = \frac{Q K^T}{2} $$
    由于 $Q$ 的维度是 $2 \times 4$，$K^T$ 的维度是 $4 \times 3$，矩阵乘法后除以 2 得到的分数矩阵 $A$ 维度为 **$2 \times 3$**。

2.  **第二步：对行进行 Softmax 归一化得到注意力权重矩阵 $S$**
    对于 $A$ 的每一行（含 3 个元素），分别做 Softmax 转换：
    $$ S_{ij} = \frac{e^{A_{ij}}}{\sum_{m=1}^3 e^{A_{im}}} $$
    归一化后的概率权重矩阵 $S$ 维度依然为 **$2 \times 3$**。

3.  **第三步：与值矩阵 $V$ 进行加权求和得到输出矩阵 $\text{Output}$**
    $$ \text{Output} = S \times V $$
    由于 $S$ 的维度是 $2 \times 3$，$V$ 的维度为 $3 \times 5$，最终相乘输出的矩阵维度为 **$2 \times 5$**。

In [6]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    """
    多头注意力 (Multi-Head Attention) 前向传播
    """
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, X):
        # X 形状: (seq_len, batch, d_model)
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性变换
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 2. 转换形状以分离多头:
        # (seq_len, batch, d_model) -> (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 1).transpose(1, 2)
        
        # 3. 计算缩放点积注意力
        # scores 形状: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = torch.softmax(scores, dim=-1)
        
        # context 形状: (batch, num_heads, seq_len, d_k)
        context = torch.matmul(attn_weights, V)
        
        # 4. 拼接多头并经过最终线性层
        # (batch, num_heads, seq_len, d_k) -> (seq_len, batch, num_heads, d_k) -> (seq_len, batch, d_model)
        context = context.transpose(1, 2).transpose(0, 1)
        context = context.contiguous().view(seq_len, batch_size, self.d_model)
        
        output = self.W_o(context)
        return output

# 验证测试
seq_len, batch, d_model = 6, 2, 4
mha = MultiHeadAttention(d_model=d_model, num_heads=2)
X = torch.randn(seq_len, batch, d_model)
output = mha(X)
print("多头注意力输出特征形状:", output.shape)  # 应该与输入 X 一致: (6, 2, 4)

多头注意力输出特征形状: torch.Size([6, 2, 4])
